In [19]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

%store -r MIMIC_DIR
%store -r cohort

In [20]:
outcomes = cohort[["subject_id", "hadm_id", "hospital_expire_flag", "deathtime", "admittime", "dischtime", "dod"]].copy() # differnet outcomes of patients

for col in ["deathtime", "admittime", "dischtime", "dod"]: 
    outcomes[col] = pd.to_datetime(outcomes[col], errors = "coerce")

outcomes["los_days"] = (outcomes.dischtime - outcomes.admittime).dt.total_seconds() / 86400

negative_los = outcomes[outcomes["los_days"] < 0]
print(f"Negative LOS rows: {len(negative_los)}")
print(negative_los[["subject_id", "hadm_id", "admittime", "dischtime", "los_days"]])

# excludes in-hospital deaths — dischtime ~= dod for those, so without this exclusion
# every hospital death would trivially also count as a "post-discharge" death
outcomes["died_within_30d_discharge"] = (
    (outcomes.hospital_expire_flag == 0) &
    outcomes.dod.notna() &
    (outcomes.dod - outcomes.dischtime).dt.days.between(0, 30)
)

# Primary study outcome (background.txt): mortality within 48 hours of admission.
# Uses deathtime (precise in-hospital timestamp) rather than dod (date-only, from
# patients.csv), since dod's day-level granularity can't reliably resolve a 48-hour
# window and in-hospital deaths always have deathtime populated.
outcomes["hours_to_death"] = (outcomes.deathtime - outcomes.admittime).dt.total_seconds() / 3600
outcomes["died_within_48h"] = (
    outcomes.deathtime.notna() &
    outcomes["hours_to_death"].between(0, 48)
)
print(f"Died within 48h of admission: {outcomes['died_within_48h'].sum():,} ({outcomes['died_within_48h'].mean():.1%})")

%store outcomes

Negative LOS rows: 29
       subject_id   hadm_id           admittime           dischtime  los_days
2109     10294074  23396294 2194-08-07 00:49:00 2194-08-07 00:00:00 -0.034028
2988     10421309  23792607 2159-04-06 20:05:00 2159-04-06 00:00:00 -0.836806
5284     10719323  22626754 2188-10-03 11:18:00 2188-10-03 00:01:00 -0.470139
7765     11042406  20609236 2131-01-31 15:08:00 2131-01-31 03:11:00 -0.497917
8901     11191729  22277446 2139-07-24 00:41:00 2139-07-24 00:01:00 -0.027778
9275     11230841  29647253 2179-04-26 04:28:00 2179-04-26 03:20:00 -0.047222
9763     11305179  24834923 2157-07-17 13:00:00 2157-07-17 02:11:00 -0.450694
14659    11968900  20087370 2170-09-02 16:09:00 2170-09-02 00:00:00 -0.672917
15658    12103788  20020878 2116-07-31 17:25:00 2116-07-31 00:00:00 -0.725694
17798    12376923  24083260 2146-12-11 10:15:00 2146-12-11 00:00:00 -0.427083
18707    12487892  22390219 2186-11-21 14:31:00 2186-11-21 01:33:00 -0.540278
26836    13535122  21247013 2177-08-15 11:

In [21]:
# ADVERSE EVENT : NALOXONE ADMINISTRATION
prescriptions = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/prescriptions.csv",
    usecols = ["subject_id", "hadm_id", "drug", "starttime", "stoptime"],
    dtype = {"subject_id": "int64", "hadm_id": "Int64"},
    parse_dates = ["starttime", "stoptime"],
)

naloxone_mask = prescriptions["drug"].str.contains("naloxone", case = False, na = False)

naloxone_counts = (
    prescriptions[naloxone_mask]
    .groupby("hadm_id").size()
    .reset_index(name = "n_naloxone_orders")
)
naloxone_flag = (
    cohort[["hadm_id"]]
    .merge(naloxone_counts, on= "hadm_id", how = "left")
)
naloxone_flag["n_naloxone_orders"] = naloxone_flag["n_naloxone_orders"].fillna(0).astype(int)
naloxone_flag["received_naloxone"] = naloxone_flag["n_naloxone_orders"] > 0
%store naloxone_flag

# Raw per-order rows (with timing) are stored separately so downstream dose-level work can
# check for naloxone in a short window *after* a specific dose, instead of using this
# whole-admission `naloxone_flag` as a predictor. Naloxone is usually a *response* to an
# opioid-related event, so admission-level use is an adverse-event/outcome signal, not a
# safe covariate for predicting mortality.
naloxone_orders = prescriptions.loc[naloxone_mask, ["subject_id", "hadm_id", "starttime", "stoptime"]].copy()
%store naloxone_orders

Stored 'naloxone_flag' (DataFrame)
Stored 'naloxone_orders' (DataFrame)


In [22]:
# ICD CODES
diagnoses = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/diagnoses_icd.csv", dtype= {"icd_code": str})
pall_icd_mask = (
    ((diagnoses.icd_version == 9) & (diagnoses.icd_code == "V667")) |
    ((diagnoses.icd_version == 10) & (diagnoses.icd_code == "Z515"))
)
pall_icd_flag = ( # gets rid of duplicates and makes sure all labeled
    diagnoses[pall_icd_mask][["hadm_id"]]
    .drop_duplicates()
    .assign(has_palliative_icd = True)
)

In [23]:
# COMFORT CARE ORDERS
palliative_flag = ( # merging palliative services
    cohort[["hadm_id"]]
    .merge(pall_icd_flag, on = "hadm_id", how = "left")
)

palliative_flag["has_palliative_icd"] = palliative_flag["has_palliative_icd"].fillna(False)

palliative_flag["any_palliative_signal"] = palliative_flag["has_palliative_icd"]

print(palliative_flag["any_palliative_signal"].mean())

%store palliative_flag

0.08047300125575554
Stored 'palliative_flag' (DataFrame)


In [24]:
services = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/services.csv", parse_dates = ["transfertime"])
print(services.curr_service.value_counts())

curr_service
MED      286528
CMED      54452
SURG      51640
OMED      33549
ORTHO     26991
NMED      26709
OBS       24923
NSURG     16795
CSURG     13421
VSURG     12385
PSYCH      9592
TRAUM      8961
GYN        7935
GU         7083
TSURG      5767
PSURG      4427
ENT        1798
DENT         59
EYE          54
NB            1
NBB           1
Name: count, dtype: int64


In [25]:
icustays = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/icustays.csv",
    usecols = ["subject_id", "hadm_id", "stay_id", "outtime", "los"]
)
icu_flag = (
    icustays.groupby("hadm_id")
    .agg(
        n_icu_stays=("stay_id", "count"),
        total_icu_los_days = ("los", "sum"),
    )
    .reset_index()
)
icu_flag["had_icu_stay"] = True

%store icu_flag
%store icustays

Stored 'icu_flag' (DataFrame)
Stored 'icustays' (DataFrame)


In [26]:
d_items = pd.read_csv("/home/emma/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv")
vent_items = d_items[d_items["label"].str.contains("invasive vent", case = False, na = False)]
vent_itemids = set(vent_items["itemid"])

proc_reader = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/procedureevents.csv",
    usecols = ["subject_id", "hadm_id", "itemid", "starttime", "endtime"],
    parse_dates = ["starttime", "endtime"],
    chunksize = 1_000_000,
)

vent_chunks = []
for chunk in proc_reader:
    chunk = chunk[chunk.hadm_id.isin(cohort.hadm_id) & chunk.itemid.isin(vent_itemids)]
    if len(chunk):
        vent_chunks.append(chunk)

vent_events = pd.concat(vent_chunks, ignore_index = True)
vent_flag = (
    vent_events.groupby("hadm_id")
    .agg(n_vent_events = ("itemid", "count"))
    .reset_index()
)

vent_flag["mechanically_ventilated"] = True

%store vent_flag
# raw timed events (with starttime/endtime), stored separately so downstream dose-level
# work can check "ventilated as of this specific dose's time" instead of the whole-admission
# vent_flag, which would otherwise leak ventilation that started after a given dose
%store vent_events


Stored 'vent_flag' (DataFrame)
Stored 'vent_events' (DataFrame)


In [27]:
vasopressor_items = d_items[
    d_items["label"].str.contains(
        "norepinephrine|epinephrine|vasopressin|dopamine|phenylephrine|dobutamine",
        case = False, na = False
    )
]
vaso_itemids = set(vasopressor_items["itemid"])

input_reader = pd.read_csv(
    "/home/emma/data/physionet.org/files/mimiciv/3.1/icu/inputevents.csv",
    usecols = ["subject_id", "hadm_id", "itemid", "starttime", "endtime"],
    parse_dates = ["starttime", "endtime"],
    chunksize = 1_000_000
)

vaso_chunks = []
for chunk in input_reader:
    chunk = chunk[chunk.hadm_id.isin(cohort.hadm_id) & chunk.itemid.isin(vaso_itemids)]
    if len(chunk):
        vaso_chunks.append(chunk)

vaso_events = pd.concat(vaso_chunks, ignore_index = True)
vaso_flag = (
    vaso_events.groupby("hadm_id")
    .agg(n_vasopressor_events = ("itemid", "count"))
    .reset_index()
)
vaso_flag["received_vasopressors"] = True

%store vaso_flag
# raw timed events, stored separately for the same reason as vent_events above --
# lets dose-level work check "on vasopressors as of this dose" instead of "ever this admission"
%store vaso_events


Stored 'vaso_flag' (DataFrame)
Stored 'vaso_events' (DataFrame)
